In [6]:
%pip install plotly
%pip install dash

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 87.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:

import pickle
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from dash import Dash, html, dcc

# -----------------------------
# Load data
# -----------------------------
with open("results.pkl", "rb") as f:
    results = pickle.load(f)

top_df = results["top_per_year"]
genre_df = results["genre_counts"]
G = results["graph"]

# -----------------------------
# 1. Time Series (binned)
# -----------------------------
top_df["year_bin"] = (top_df["startYear"] // 5) * 5

binned = top_df.loc[
    top_df.groupby("year_bin")["averageRating"].idxmax()
].sort_values("year_bin")

fig_time = px.line(
    binned,
    x="year_bin",
    y="averageRating",
    markers=True,
    hover_data=["primaryTitle"],
    title="Top Fantasy Movies Over Time"
)

fig_time.update_layout(
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Rating"
)

# -----------------------------
# 2. Genre Bar Chart
# -----------------------------
fig_genre = px.bar(
    genre_df.head(10),
    x="Genre",
    y="Count",
    title="Top Genre Pairings with Fantasy"
)

fig_genre.update_layout(template="plotly_white")

# -----------------------------
# 3. Network Graph
# -----------------------------
pos = nx.spring_layout(G, seed=42)

edge_x = []
edge_y = []

for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5),
    hoverinfo='none',
    mode='lines'
)

node_x = []
node_y = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(size=6),
    text=list(G.nodes()),
    hoverinfo='text'
)

fig_graph = go.Figure(data=[edge_trace, node_trace])
fig_graph.update_layout(
    title="Actor-Director Network",
    showlegend=False,
    template="plotly_white"
)

# -----------------------------
# DASH APP
# -----------------------------
app = Dash(__name__)

app.layout = html.Div([
    html.H1("IMDB Fantasy Dashboard", style={"textAlign": "center"}),

    dcc.Graph(figure=fig_time),

    dcc.Graph(figure=fig_genre),

    dcc.Graph(figure=fig_graph),
])

# -----------------------------
# Run server
# -----------------------------
if __name__ == "__main__":
    app.run(debug=True)